In [6]:
# CELL 1 — Reload everything cleanly
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df_blinkit   = pd.read_excel('../data/raw/BlinkIT Grocery Data.csv')
df_zepto     = pd.read_excel('../data/raw/zepto_v1.csv')
df_groceries = pd.read_csv('../data/raw/Groceries_dataset.csv', encoding='latin-1', engine='python')
df_bigbasket = pd.read_csv('../data/raw/BigBasket Products.csv', encoding='latin-1', on_bad_lines='skip', engine='python')

print('✅ All loaded')
print(f'Blinkit: {df_blinkit.shape} | Zepto: {df_zepto.shape}')
print(f'Groceries: {df_groceries.shape} | BigBasket: {df_bigbasket.shape}')

✅ All loaded
Blinkit: (8523, 12) | Zepto: (3732, 9)
Groceries: (38765, 3) | BigBasket: (27555, 10)


In [7]:
# CELL 2 — Clean Blinkit
df_blinkit_clean = df_blinkit.copy()

# Standardize column names
df_blinkit_clean.columns = df_blinkit_clean.columns.str.strip().str.lower().str.replace(' ', '_')

# Rename to standard names
df_blinkit_clean = df_blinkit_clean.rename(columns={
    'item_type': 'category',
    'sales': 'sale_price',
    'outlet_type': 'outlet_type'
})

# Add platform column
df_blinkit_clean['platform'] = 'Blinkit'

# Drop nulls in key columns
df_blinkit_clean = df_blinkit_clean.dropna(subset=['category', 'sale_price'])

print('✅ Blinkit cleaned')
print(f'Shape: {df_blinkit_clean.shape}')
print(df_blinkit_clean[['category', 'sale_price', 'platform']].head(3))

✅ Blinkit cleaned
Shape: (8523, 13)
                category  sale_price platform
0  Fruits and Vegetables    145.4786  Blinkit
1     Health and Hygiene    115.3492  Blinkit
2           Frozen Foods    165.0210  Blinkit


In [8]:
# CELL 3 — Clean Zepto
df_zepto_clean = df_zepto.copy()

# Standardize column names
df_zepto_clean.columns = df_zepto_clean.columns.str.strip().str.lower()

# Rename to standard names
df_zepto_clean = df_zepto_clean.rename(columns={
    'name': 'product',
    'mrp': 'market_price',
    'discountedselling price': 'sale_price',
    'discountedsellingprice': 'sale_price',
    'discountpercent': 'discount_pct',
    'category': 'category'
})

# Add platform column
df_zepto_clean['platform'] = 'Zepto'

# Drop out of stock items
df_zepto_clean = df_zepto_clean[df_zepto_clean['outofstock'] == False] if 'outofstock' in df_zepto_clean.columns else df_zepto_clean

print('✅ Zepto cleaned')
print(f'Shape: {df_zepto_clean.shape}')
print(df_zepto_clean.head(3))

✅ Zepto cleaned
Shape: (3279, 10)
              category         product  market_price  discount_pct  \
0  Fruits & Vegetables           Onion          2500            16   
1  Fruits & Vegetables   Tomato Hybrid          4200            16   
2  Fruits & Vegetables  Tender Coconut          5100            15   

   availablequantity  sale_price  weightingms  outofstock  quantity platform  
0                  3        2100         1000       False         1    Zepto  
1                  3        3500         1000       False         1    Zepto  
2                  3        4300           58       False         1    Zepto  


In [9]:
# CELL 4 — Clean BigBasket
df_bigbasket_clean = df_bigbasket.copy()

# Rename to standard names
df_bigbasket_clean = df_bigbasket_clean.rename(columns={
    'product': 'product',
    'category': 'category',
    'sale_price': 'sale_price',
    'market_price': 'market_price'
})

# Add platform column
df_bigbasket_clean['platform'] = 'BigBasket'

# Drop nulls
df_bigbasket_clean = df_bigbasket_clean.dropna(subset=['sale_price', 'category'])

# Compute discount %
df_bigbasket_clean['discount_pct'] = ((df_bigbasket_clean['market_price'] - df_bigbasket_clean['sale_price']) / df_bigbasket_clean['market_price'] * 100).round(1)

print('✅ BigBasket cleaned')
print(f'Shape: {df_bigbasket_clean.shape}')
print(df_bigbasket_clean[['product','category','sale_price','market_price','discount_pct']].head(3))

✅ BigBasket cleaned
Shape: (27555, 12)
                                  product                category  sale_price  \
0  Garlic Oil - Vegetarian Capsule 500 mg        Beauty & Hygiene       220.0   
1                   Water Bottle - Orange  Kitchen, Garden & Pets       180.0   
2          Brass Angle Deep - Plain, No.2    Cleaning & Household       119.0   

   market_price  discount_pct  
0         220.0           0.0  
1         180.0           0.0  
2         250.0          52.4  


In [10]:
# CELL 5 — Clean Groceries (your RFM + basket data)
df_groceries_clean = df_groceries.copy()

# Parse date
df_groceries_clean['Date'] = pd.to_datetime(df_groceries_clean['Date'], dayfirst=True)

# Rename columns
df_groceries_clean = df_groceries_clean.rename(columns={
    'Member_number': 'customer_id',
    'Date': 'order_date',
    'itemDescription': 'product'
})

# Sort by customer and date
df_groceries_clean = df_groceries_clean.sort_values(['customer_id', 'order_date']).reset_index(drop=True)

# Add first order date per customer
df_groceries_clean['first_order_date'] = df_groceries_clean.groupby('customer_id')['order_date'].transform('min')

print('✅ Groceries cleaned')
print(f'Shape: {df_groceries_clean.shape}')
print(f'Date range: {df_groceries_clean["order_date"].min()} → {df_groceries_clean["order_date"].max()}')
print(f'Unique customers: {df_groceries_clean["customer_id"].nunique()}')
print(f'Unique products: {df_groceries_clean["product"].nunique()}')
print(df_groceries_clean.head(5))

✅ Groceries cleaned
Shape: (38765, 4)
Date range: 2014-01-01 00:00:00 → 2015-12-30 00:00:00
Unique customers: 3898
Unique products: 167
   customer_id order_date      product first_order_date
0         1000 2014-06-24   whole milk       2014-06-24
1         1000 2014-06-24       pastry       2014-06-24
2         1000 2014-06-24  salty snack       2014-06-24
3         1000 2015-03-15      sausage       2014-06-24
4         1000 2015-03-15   whole milk       2014-06-24


In [11]:
import os
os.makedirs('../data/clean', exist_ok=True)
print('✅ Clean folder created')

✅ Clean folder created


In [12]:
# CELL 6 — Save all clean files
df_blinkit_clean.to_csv('../data/clean/blinkit_clean.csv', index=False)
df_zepto_clean.to_csv('../data/clean/zepto_clean.csv', index=False)
df_bigbasket_clean.to_csv('../data/clean/bigbasket_clean.csv', index=False)
df_groceries_clean.to_csv('../data/clean/groceries_clean.csv', index=False)

print('✅ DAY 2 COMPLETE!')
print('='*50)
print('All 4 clean files saved to data/clean/')
print(f'Blinkit:   {df_blinkit_clean.shape}')
print(f'Zepto:     {df_zepto_clean.shape}')
print(f'BigBasket: {df_bigbasket_clean.shape}')
print(f'Groceries: {df_groceries_clean.shape}')
print('='*50)
print('Day 3 tomorrow: Price Intelligence Matrix 🔥')

✅ DAY 2 COMPLETE!
All 4 clean files saved to data/clean/
Blinkit:   (8523, 13)
Zepto:     (3279, 10)
BigBasket: (27555, 12)
Groceries: (38765, 4)
Day 3 tomorrow: Price Intelligence Matrix 🔥
